# Parking Detection and Plate Recognition

This notebook combines two YOLOv8n models for a Malaysian car park:

- **Parking detection** (`parking_detection_yolov8n.pt`): finds empty parking slots, parked cars and illegally parked cars (cars double-parked in the driving aisle). It is trained in Steps 1-17 (skipped when the model file is already there).
- **Plate recognition** (`license_plate_yolov8n.pt` + EasyOCR): finds and reads the licence plate of every illegally parked car (Step 18). It is trained in `base_models/PlateRecognition.ipynb`.

On Colab, drag `datasets.zip` into the Files panel and run all cells. The zip is the `datasets` folder, which holds the dataset and both trained models in `datasets/models`, so nothing else has to be uploaded.

## Reminder

The model finds illegally parked cars by itself, so no setup is needed for drone footage.

For a fixed camera, the code in Step 14 can also check every car against the parking slot boundaries (a car over two slots, or over the slot boundary):

1. Go to Step 14, "Define the parking slot boundaries".
2. Fill in `SLOTS` with the 4 corner points (x, y) of every parking slot in your camera image.
3. Run "Check for illegal parking".

A drone moves, so its slots are in a different place in every photo. Leave `SLOTS` empty for drone footage.

## Step 1: Install the libraries and load the dataset

The dataset and both trained models come from `datasets.zip`. On Colab, drag it into the Files panel on the left before running this cell. When running locally, keep it next to this notebook.

In [ ]:
!pip install -q ultralytics easyocr

import os
import zipfile

# datasets.zip is the "datasets" folder zipped (on Colab: drag it into the Files panel on the left).
# It holds the dataset and the trained models. If the folder is already here, the zip is not needed.
if not os.path.exists("datasets"):
    with zipfile.ZipFile("datasets.zip") as z:
        z.extractall(".")

# Both trained models are kept here: parking_detection_yolov8n.pt and license_plate_yolov8n.pt
MODEL_DIR = os.path.join("datasets", "models")

path = os.path.abspath(".")

## Step 2: Show the dataset path

In [ ]:
print("Dataset path:")
print(path)

### Show the dataset folders

In [ ]:
import os

print(os.listdir(path))

## Step 3: Find the dataset folder automatically

In [ ]:
DATASET_PATH = None

for root, dirs, files in os.walk(path):
    if "carpark-and-lp-with-illegal-parking" in dirs:
        DATASET_PATH = os.path.join(root, "carpark-and-lp-with-illegal-parking")
        break

print("Dataset path:")
print(DATASET_PATH)

## Step 4: Check the classes and number of images

The dataset has drone photos of a Malaysian car park with four classes. Three are used:

- Empty: a free parking slot
- Occupied: a parked car in a slot
- Illegal: a car double-parked in the driving aisle

The fourth class, LicensePlate, is not used here (licence plates are found by the separate plate model in Step 18).

In [ ]:
import json
from collections import Counter

SPLITS = ["train", "valid", "test"]
class_names = ["Empty", "Occupied", "Illegal"]

coco = {}
for split in SPLITS:
    with open(os.path.join(DATASET_PATH, split, "_annotations.coco.json")) as f:
        coco[split] = json.load(f)

# COCO category id mapped to our class name, matched by name (not by id).
# Categories not listed (carpark, LicensePlate) are ignored.
CLASS_MAP = {c["id"]: c["name"] for c in coco["train"]["categories"] if c["name"] in class_names}

print("Categories in the dataset:")
for c in coco["train"]["categories"]:
    print(f"  {c['id']}: {c['name']}")

print("\nClasses:")
print(class_names)

print("\nNumber of images and objects:")
for split in SPLITS:
    counts = Counter(CLASS_MAP.get(a["category_id"], "ignored") for a in coco[split]["annotations"])
    print(f"{split}: {len(coco[split]['images'])} images, {dict(counts)}")

# Every class must have training boxes, otherwise the dataset is the wrong one
train_counts = Counter(CLASS_MAP.get(a["category_id"]) for a in coco["train"]["annotations"])
missing = [name for name in class_names if train_counts[name] == 0]
assert not missing, f"No training boxes for {missing}: check that this is the right dataset"

print("\nTotal number of classes:", len(class_names))

## Step 5: Display sample images from each class

Green boxes are empty slots, blue boxes are parked cars and red boxes are illegally parked cars.

In [ ]:
import random
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

COLORS = {"Empty": "lime", "Occupied": "dodgerblue", "Illegal": "red"}
data = coco["train"]

# Find every image that contains each class
images_with_class = {name: [] for name in class_names}
for img_info in data["images"]:
    names = {CLASS_MAP.get(a["category_id"]) for a in data["annotations"] if a["image_id"] == img_info["id"]}
    for name in names & set(class_names):
        images_with_class[name].append(img_info)

plt.figure(figsize=(15, 4 * len(class_names)))

for row, class_name in enumerate(class_names):
    for col, img_info in enumerate(random.sample(images_with_class[class_name], 3)):
        ax = plt.subplot(len(class_names), 3, row * 3 + col + 1)
        ax.imshow(Image.open(os.path.join(DATASET_PATH, "train", img_info["file_name"])).convert("RGB"))

        # Only draw the boxes of this class
        for a in data["annotations"]:
            if a["image_id"] == img_info["id"] and CLASS_MAP.get(a["category_id"]) == class_name:
                x, y, w, h = a["bbox"]
                ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, color=COLORS[class_name], linewidth=2))

        plt.title(f"{class_name} (Sample {col + 1})")
        plt.axis("off")

plt.tight_layout()
plt.show()

## Step 6: Load the dataset in YOLO format

YOLOv8 reads the images from folders, but every box must be written in its own format: one text file per image, with one line per object.

- Every image is resized to 512 x 512 pixels.
- Images are grouped into batches of 16.
- The data is already split into training, validation and test data, so the split is kept.

The converted dataset is saved in `datasets/carpark-yolo`.

In [ ]:
import shutil

# The converted dataset is saved in datasets/carpark-yolo
YOLO_PATH = os.path.join(os.path.dirname(DATASET_PATH), "carpark-yolo")

if os.path.exists(YOLO_PATH):
    shutil.rmtree(YOLO_PATH)  # start fresh every time

for split in SPLITS:
    data = coco[split]
    img_dir = os.path.join(YOLO_PATH, split, "images")
    lbl_dir = os.path.join(YOLO_PATH, split, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    for img_info in data["images"]:
        W, H = img_info["width"], img_info["height"]

        # YOLO format: one line per object: class_id x_center y_center width height (values 0-1)
        lines = []
        for a in data["annotations"]:
            if a["image_id"] != img_info["id"] or a["category_id"] not in CLASS_MAP:
                continue
            x, y, w, h = a["bbox"]
            class_id = class_names.index(CLASS_MAP[a["category_id"]])
            lines.append(f"{class_id} {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")

        shutil.copy(os.path.join(DATASET_PATH, split, img_info["file_name"]), img_dir)
        with open(os.path.join(lbl_dir, os.path.splitext(img_info["file_name"])[0] + ".txt"), "w") as f:
            f.write("\n".join(lines))

    print(f"{split}: {len(os.listdir(img_dir))} images")

## Step 7: Display one training batch

In [ ]:
import matplotlib.pyplot as plt

BATCH_SIZE = 16
train_img_dir = os.path.join(YOLO_PATH, "train", "images")
train_lbl_dir = os.path.join(YOLO_PATH, "train", "labels")

plt.figure(figsize=(10, 10))

for i, file_name in enumerate(random.sample(os.listdir(train_img_dir), 9)):
    ax = plt.subplot(3, 3, i + 1)
    image = Image.open(os.path.join(train_img_dir, file_name)).convert("RGB")
    W, H = image.size
    ax.imshow(image)

    with open(os.path.join(train_lbl_dir, os.path.splitext(file_name)[0] + ".txt")) as f:
        for line in f:
            if not line.strip():
                continue
            class_id, x, y, w, h = line.split()
            name = class_names[int(class_id)]
            x, y, w, h = float(x) * W, float(y) * H, float(w) * W, float(h) * H
            ax.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h, fill=False, color=COLORS[name], linewidth=2))
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 8: Create the dataset file

`data.yaml` tells YOLO:

- `train`, `val` and `test`: where the training, validation and test images are
- `nc`: the number of classes
- `names`: the class names

In [ ]:
DATA_YAML = os.path.join(YOLO_PATH, "data.yaml")

with open(DATA_YAML, "w") as f:
    f.write(f"path: {YOLO_PATH}\n")
    f.write("train: train/images\nval: valid/images\ntest: test/images\n")
    f.write(f"nc: {len(class_names)}\n")
    f.write(f"names: {class_names}\n")

print(open(DATA_YAML).read())
print("Dataset pipeline is ready.")

## Step 9: Build the AI model (YOLOv8n)

This is the AI model that will detect the cars and parking slots in our images. Instead of creating a CNN from scratch, we use transfer learning: YOLOv8n was already trained on the COCO dataset, so it starts with useful knowledge of cars and objects.

The model has three parts. The backbone, pre-trained on COCO, takes the 512x512 input image and extracts features such as edges, wheels and car shapes. The neck combines small and large features so that cars both near and far from the camera are found. The detection head then predicts a box, a class (Empty, Occupied or Illegal) and a confidence score for every object.

Why YOLOv8n:

- Very fast
- Small model size
- Designed for edge devices
- High accuracy
- Compatible with TensorFlow Lite
- Can be compiled for the Coral Edge TPU

In [ ]:
from ultralytics import YOLO

# Load the pre-trained YOLOv8n model
model = YOLO("yolov8n.pt")

# Show model architecture
model.info()

## Step 10: Set the training settings

YOLO has no separate `compile()` step like TensorFlow. The optimizer, number of epochs and image size are chosen here and passed to training in the next step. The loss function and the metrics (precision, recall and mAP) are built in.

In [ ]:
import torch

# YOLO has no compile() step like TensorFlow, the settings are chosen here instead.
# The loss function (box loss + class loss) and the metrics (precision, recall, mAP) are built in.
TRAIN_SETTINGS = dict(
    data=DATA_YAML,
    epochs=60,
    patience=20,        # stop early if validation stops improving for 20 epochs
    imgsz=512,
    batch=BATCH_SIZE,
    optimizer="auto",   # YOLO picks the best optimizer
    device=0 if torch.cuda.is_available() else "cpu",   # use a GPU: on Colab choose Runtime, Change runtime type, T4 GPU
    project="runs",
    name="parking",
    exist_ok=True,
)

print("Model settings are ready.")

## Step 11: Train the AI model

In this step, the AI learns from the training images.

- Training dataset: used to teach the model.
- Validation dataset: used to evaluate the model after each epoch.
- Epochs = 60: the model sees the entire dataset up to 60 times.

During training, YOLO displays something like:

```text
Epoch 1/60
box_loss: 1.52
cls_loss: 2.10
Precision: 0.65
Recall: 0.58
mAP50: 0.61
```

Each epoch shows:

- box_loss: box position error (lower is better)
- cls_loss: class error (lower is better)
- Precision: how many detected boxes are correct
- Recall: how many real objects were found
- mAP50: detection accuracy (higher is better)

At each epoch, mAP50 should increase and the losses should decrease. If the final mAP50 is above 0.70, the model is performing well.

Use a GPU for training (on Colab: Runtime, Change runtime type, T4 GPU). On a CPU it is much slower.

In [ ]:
import shutil

# Already trained before? Keep parking_detection_yolov8n.pt in datasets/models and training is skipped.
# Delete it to train again.
SAVED_MODEL = os.path.join(MODEL_DIR, "parking_detection_yolov8n.pt")
RUN_DIR = None

# A saved model is only reused if it was trained with the same classes
if os.path.exists(SAVED_MODEL) and list(YOLO(SAVED_MODEL).names.values()) == class_names:
    print(f"Found {SAVED_MODEL}, skipping training.")
else:
    results = model.train(**TRAIN_SETTINGS)
    RUN_DIR = str(results.save_dir)
    shutil.copy(os.path.join(RUN_DIR, "weights", "best.pt"), SAVED_MODEL)
    print("Training results saved in:", RUN_DIR)

## Step 12: Plot training accuracy and loss

mAP50 is the detector's version of accuracy, so it is plotted together with precision and recall.

In [ ]:
# The training history only exists if the model was trained in this session
if RUN_DIR is None:
    print("Training was skipped, so there is no training history to plot.")
else:
    import pandas as pd
    import matplotlib.pyplot as plt

    history = pd.read_csv(os.path.join(RUN_DIR, "results.csv"))
    history.columns = history.columns.str.strip()

    # Accuracy values (mAP50 is the "accuracy" of a detector)
    val_map50 = history["metrics/mAP50(B)"]
    val_precision = history["metrics/precision(B)"]
    val_recall = history["metrics/recall(B)"]

    # Loss values
    train_loss = history["train/box_loss"] + history["train/cls_loss"]
    val_loss = history["val/box_loss"] + history["val/cls_loss"]

    epochs_range = history["epoch"]

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, val_map50, label="Validation mAP50")
    plt.plot(epochs_range, val_precision, label="Validation Precision")
    plt.plot(epochs_range, val_recall, label="Validation Recall")
    plt.title("Validation Accuracy (mAP50)")
    plt.xlabel("Epoch")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, train_loss, label="Training Loss")
    plt.plot(epochs_range, val_loss, label="Validation Loss")
    plt.title("Training and Validation Loss")
    plt.xlabel("Epoch")
    plt.legend()

    plt.show()

## Step 13: Evaluate the model

This is the final performance of the AI model, measured on test images it has never seen before.

There are three important metrics:

- mAP50: how many objects were detected correctly. For example, mAP50 = 0.85 means the model correctly found about 85 out of every 100 objects. Higher is better.
- Precision: how many of the detected boxes are correct. Higher is better.
- Recall: how many of the real objects were found. Higher is better.

If the test mAP50 is high, the model can generalise well to new images.

In [ ]:
# Evaluate the model on the test dataset
best_model = YOLO(SAVED_MODEL)
metrics = best_model.val(data=DATA_YAML, split="test", imgsz=512)

print("\n========== Model Evaluation ==========")
print(f"{'Class':<10}{'Precision':>10}{'Recall':>10}{'mAP50':>10}")
for i, class_id in enumerate(metrics.box.ap_class_index):
    p, r, ap50, _ = metrics.box.class_result(i)
    print(f"{class_names[class_id]:<10}{p:>10.3f}{r:>10.3f}{ap50:>10.3f}")
print(f"{'ALL':<10}{metrics.box.mp:>10.3f}{metrics.box.mr:>10.3f}{metrics.box.map50:>10.3f}")

## Step 14: Test the AI model on one image

This is the inference stage. The model is no longer learning; it is making predictions on a new image.

The input image is resized to 512x512 and passed through the YOLOv8n model, which returns a bounding box, a predicted class and a confidence score for every object it finds.

By default a random test photo with an illegally parked car is used (a different one every time this cell runs). To use your own image on Colab, set `UPLOAD_OWN_IMAGE = True`.

In [ ]:
# False: use a random test photo that contains an illegally parked car
# True:  upload your own image on Colab
UPLOAD_OWN_IMAGE = False

if UPLOAD_OWN_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    img_path = list(uploaded.keys())[0]
else:
    test_data = coco["test"]
    illegal_ids = {a["image_id"] for a in test_data["annotations"] if CLASS_MAP.get(a["category_id"]) == "Illegal"}
    img_info = random.choice([i for i in test_data["images"] if i["id"] in illegal_ids])  # a different photo every run
    img_path = os.path.join(DATASET_PATH, "test", img_info["file_name"])

print("Image:", img_path)

In [ ]:
CONF_THRESHOLD = 0.35

result = best_model.predict(img_path, conf=CONF_THRESHOLD, imgsz=512, verbose=False)[0]

plt.figure(figsize=(7, 7))
ax = plt.gca()
ax.imshow(Image.open(img_path).convert("RGB"))

counts = Counter()
for box in result.boxes:
    name = class_names[int(box.cls)]
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=COLORS[name], linewidth=2))
    ax.text(x1, y1 - 4, f"{name} {float(box.conf) * 100:.2f}%", color=COLORS[name], fontsize=9, weight="bold")
    counts[name] += 1

plt.title(f"Cars: {counts['Occupied'] + counts['Illegal']} | Empty: {counts['Empty']} | Illegal: {counts['Illegal']}")
plt.axis("off")
plt.show()

for name in class_names:
    print(f"{name}: {counts[name]}")

### Define the parking slot boundaries

Only for a fixed camera: it does not move, so the parking slots are drawn only once. Each slot is 4 corner points (x, y) in the image.

A drone moves, so `SLOTS` is left empty and only the model's Illegal class is used.

In [ ]:
# Fixed camera: the 4 corner points (x, y) of every parking slot, for example
#   "P01": [(18, 120), (112, 120), (112, 445), (18, 445)],
# Drone footage: leave it empty (the slots move in every photo)
SLOTS = {}

plt.figure(figsize=(7, 7))
ax = plt.gca()
ax.imshow(Image.open(img_path).convert("RGB"))
for slot_id, corners in SLOTS.items():
    ax.add_patch(patches.Polygon(corners, fill=False, color="yellow", linewidth=2, linestyle="--"))
    ax.text(corners[0][0] + 5, corners[0][1] - 8, slot_id, color="yellow", fontsize=11, weight="bold")
plt.title("Parking slots")
plt.axis("off")
plt.show()

### Check for illegal parking

For every detected car, the code measures how much of the car's footprint (the part touching the ground) lies inside each slot:

- At least 10% of the footprint in two or more slots: overlaps multiple slots (for example P02 + P03).
- At least 10% of the footprint outside all slots: crosses the parking boundary.
- Otherwise: properly parked.

**Footprint.** Seen from the side, the top of a car leans over the slot behind it, so the whole box would wrongly overlap two slots. `FOOTPRINT_SHARE` sets how much of the box, measured from its bottom edge, counts as the footprint:

- `1.0` for a camera looking straight down (like drone photos): the whole box is the footprint.
- About `0.35` for an angled camera: only the bottom part of the box, where the wheels are.

A car is a violation if the model classified it as Illegal, or (when `SLOTS` is filled in) if this rule flags it. The penalty is simulated.

In [ ]:
import cv2
import numpy as np
import pandas as pd
from datetime import datetime

# A car counts as being in a slot if at least 10% of its footprint is inside it
OVERLAP_THRESHOLD = 0.10

# Share of the car box (from the bottom edge) that touches the ground.
# 1.0 = camera looking straight down (drone), about 0.35 = angled camera. Tune it for your camera.
FOOTPRINT_SHARE = 1.0
PENALTY = "RM 50 (simulated summons)"


def overlap_area(polygon_a, polygon_b):
    area, _ = cv2.intersectConvexConvex(np.array(polygon_a, np.float32), np.array(polygon_b, np.float32))
    return area


def check_parking(car_box, slots):
    if not slots:
        return [], "No slots defined"

    x1, y1, x2, y2 = car_box
    y1 = y2 - (y2 - y1) * FOOTPRINT_SHARE  # keep only the bottom part of the box
    car_polygon = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
    car_area = (x2 - x1) * (y2 - y1)

    share_in_slot = {slot_id: overlap_area(car_polygon, corners) / car_area for slot_id, corners in slots.items()}
    used_slots = [slot_id for slot_id, share in share_in_slot.items() if share >= OVERLAP_THRESHOLD]
    outside_share = 1 - sum(share_in_slot.values())

    if len(used_slots) >= 2:
        return used_slots, "Overlaps multiple slots"
    if outside_share >= OVERLAP_THRESHOLD:
        return used_slots, "Crosses parking boundary"
    return used_slots, "Properly parked"


detected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

plt.figure(figsize=(8, 8))
ax = plt.gca()
ax.imshow(Image.open(img_path).convert("RGB"))
for slot_id, corners in SLOTS.items():
    ax.add_patch(patches.Polygon(corners, fill=False, color="yellow", linewidth=1.5, linestyle="--"))
    ax.text(corners[0][0] + 5, corners[0][1] - 8, slot_id, color="yellow", fontsize=10, weight="bold")

rows = []
for box in result.boxes:
    name = class_names[int(box.cls)]
    if name == "Empty":
        continue  # only cars are checked
    car_box = box.xyxy[0].tolist()
    used_slots, rule_status = check_parking(car_box, SLOTS)

    # Violation if the AI model classified it as Illegal OR the slot rule flags it
    rule_violation = rule_status in ("Overlaps multiple slots", "Crosses parking boundary")
    violation = name == "Illegal" or rule_violation
    reason = "AI model: Illegal (double-parked)" if name == "Illegal" else (rule_status if rule_violation else "")
    vehicle = f"Car {len(rows) + 1:02d}"
    rows.append({
        "Vehicle": vehicle,
        "Parking Slots": " + ".join(used_slots) if used_slots else "-",
        "Status": "Parking Violation" if violation else "Properly Parked",
        "Reason": reason,
        "Detected At": detected_at,
        "Penalty": PENALTY if violation else "-",
    })

    x1, y1, x2, y2 = car_box
    color = "red" if violation else "lime"
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, linewidth=3))
    ax.text(x1, y2 + 14, vehicle, color=color, fontsize=10, weight="bold")

plt.axis("off")
plt.show()

report = pd.DataFrame(rows, columns=["Vehicle", "Parking Slots", "Status", "Reason", "Detected At", "Penalty"])
report

## Step 15: Save the trained model

At this point training is finished. The model is saved so it can be used later without training again.

The `.pt` file contains:

- the model architecture
- the learned weights
- the class names

In [ ]:
# The trained model was already saved in datasets/models in the training step.
# On Colab, download it now (the file is deleted when the runtime resets) and put it in datasets/models before zipping.
try:
    from google.colab import files
    files.download(SAVED_MODEL)
except ImportError:
    print("Model saved as", SAVED_MODEL)

## Step 16: Check that the model was saved

List the files in the current folder to make sure the model file was created.

In [ ]:
import os

print(os.listdir(MODEL_DIR))

## Step 17: Measure the inference time

Run the saved model on the same image and measure how long one prediction takes.

In [ ]:
import time
import numpy as np

saved_model = YOLO(SAVED_MODEL)
print(f"Processing image: {img_path}")

saved_model.predict(img_path, imgsz=512, verbose=False)  # warm-up run (the first run is always slower)

start_time = time.perf_counter()
result = saved_model.predict(img_path, conf=CONF_THRESHOLD, imgsz=512, verbose=False)[0]
end_time = time.perf_counter()

inference_time = (end_time - start_time) * 1000

print("========== COLAB INFERENCE ==========")
print(f"Objects detected : {len(result.boxes)}")
print(f"Inference Time   : {inference_time:.2f} ms")

### More accurate timing measurement

A single measurement is not reliable. The same image is run 20 times and the average time is calculated.

In [ ]:
import time
import numpy as np

times = []

for _ in range(20):
    start_time = time.perf_counter()

    saved_model.predict(img_path, imgsz=512, verbose=False)

    end_time = time.perf_counter()

    times.append((end_time - start_time) * 1000)

average_time = np.mean(times)

print("========== COLAB INFERENCE RESULTS ==========")
print(f"Average Inference Time : {average_time:.2f} ms")
print(f"Fastest Time           : {np.min(times):.2f} ms")
print(f"Slowest Time           : {np.max(times):.2f} ms")
print(f"Frames Per Second      : {1000 / average_time:.1f} FPS")

## Step 18: Parking detection and plate recognition

Everything needed to use both trained models, without the training steps above:

1. `detect_parking(image)`: the parking model. Returns one dict per object with `class` ("Empty", "Occupied" or "Illegal"), `conf` and `box`.
2. The OCR functions from `PlateRecognition.ipynb`: EasyOCR reads the characters and the result is corrected to the Malaysian plate format.
3. `read_plate_number(image)`: the plate model plus OCR. Returns `plate`, `raw_text`, `ocr_conf` and `box`.

The two models are kept separate on purpose. The parking model only knows Empty / Occupied / Illegal, and the plate model only knows LicensePlate, so a plate can never be classified as an illegally parked car.

Both functions take the image as an RGB array (`load_rgb(path)`) and give boxes as `(x1, y1, x2, y2)` in pixels of that image, so a car box from `detect_parking` is cut out of the image and passed to `read_plate_number`.

In [ ]:
# Parking detection: uses only parking_detection_yolov8n.pt
from ultralytics import YOLO
import numpy as np
from PIL import Image

PARKING_MODEL_FILE = os.path.join(MODEL_DIR, "parking_detection_yolov8n.pt")
PARKING_IMGSZ = 512
PARKING_CONF = 0.35

parking_model = YOLO(PARKING_MODEL_FILE)
PARKING_CLASSES = list(parking_model.names.values())  # ["Empty", "Occupied", "Illegal"]


def load_rgb(image_path):
    return np.array(Image.open(image_path).convert("RGB"))


def detect_parking(image):
    """image: RGB array. Returns one dict per object found:
    class ("Empty", "Occupied" or "Illegal"), conf (0-1) and box (x1, y1, x2, y2) in pixels of this image."""
    result = parking_model.predict(image[:, :, ::-1], conf=PARKING_CONF, imgsz=PARKING_IMGSZ, verbose=False)[0]  # YOLO expects BGR arrays
    return [{"class": PARKING_CLASSES[int(b.cls)], "conf": float(b.conf), "box": tuple(b.xyxy[0].tolist())}
            for b in result.boxes]


# Quick check on the Step 14 image
for obj in detect_parking(load_rgb(img_path)):
    print(f"{obj['class']:<9} {obj['conf'] * 100:5.1f}%  box {tuple(round(v) for v in obj['box'])}")

### Plate recognition

The plate model only finds where the plate is; EasyOCR reads it. EasyOCR downloads its own model the first time it runs.

In [ ]:
# OCR functions (from PlateRecognition.ipynb, Step 14)
import torch
import re
import numpy as np
import easyocr

reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)

PLATE_FORMAT = re.compile(r"^[A-Z]{1,3}[0-9]{1,4}[A-Z]?$")
TO_LETTER = {"0": "O", "1": "I", "2": "Z", "5": "S", "6": "G", "8": "B"}
TO_DIGIT = {"O": "0", "D": "0", "Q": "0", "I": "1", "L": "1", "Z": "2", "S": "5", "G": "6", "B": "8", "T": "7"}


def read_plate(crop):
    pieces = reader.readtext(crop, allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
    if not pieces:
        return "", 0.0

    # Put the text pieces in reading order: top line first, then left to right
    pieces.sort(key=lambda p: (p[0][0][1] + p[0][2][1]) / 2)
    lines = []
    for p in pieces:
        center_y = (p[0][0][1] + p[0][2][1]) / 2
        height = p[0][2][1] - p[0][0][1]
        if lines and abs(center_y - lines[-1][0]) < height / 2:
            lines[-1][1].append(p)
        else:
            lines.append([center_y, [p]])
    ordered = [p for _, line in lines for p in sorted(line, key=lambda p: p[0][0][0])]

    text = "".join(p[1] for p in ordered)
    confidence = float(np.mean([p[2] for p in ordered]))
    return text, confidence


def clean_plate(text):
    text = re.sub(r"[^A-Z0-9]", "", text.upper())

    # Try every way to split the text into letters + numbers + optional letter,
    # fix look-alike characters, and keep the split that needs the fewest changes
    best, best_changes = text, None
    for n_prefix in (3, 2, 1):
        for n_suffix in (0, 1):
            prefix = text[:n_prefix]
            digits = text[n_prefix:len(text) - n_suffix]
            suffix = text[len(text) - n_suffix:]

            new_prefix = "".join(TO_LETTER.get(c, c) for c in prefix)
            new_digits = "".join(TO_DIGIT.get(c, c) for c in digits)
            new_suffix = "".join(TO_LETTER.get(c, c) for c in suffix)
            changes = sum(a != b for a, b in zip(prefix + digits + suffix, new_prefix + new_digits + new_suffix))

            # Malaysian plate numbers never start with 0 (an extra 0 is usually the edge of the plate)
            changes += len(new_digits) - len(new_digits.lstrip("0"))
            new_digits = new_digits.lstrip("0")

            candidate = new_prefix + new_digits + new_suffix
            if PLATE_FORMAT.match(candidate) and (best_changes is None or changes < best_changes):
                best, best_changes = candidate, changes
    return best


def format_plate(plate):
    # VBM6529 -> VBM 6529, WC679T -> WC 679 T
    match = re.match(r"^([A-Z]+)([0-9]+)([A-Z]?)$", plate)
    return " ".join(part for part in match.groups() if part) if match else plate

In [ ]:
# Plate recognition: uses license_plate_yolov8n.pt, the OCR functions above and load_rgb() from the parking cell
from ultralytics import YOLO
import numpy as np
from PIL import Image

PLATE_MODEL_FILE = os.path.join(MODEL_DIR, "license_plate_yolov8n.pt")
PLATE_IMGSZ = 512
PLATE_CONF = 0.25
PLATE_NMS_IOU = 0.3

plate_model = YOLO(PLATE_MODEL_FILE)


def read_plate_number(image):
    """image: RGB array of a car (whole photo or a crop of one car).
    Finds the most confident plate and reads it. Returns a dict:
    plate ("VBM 6529", or "" if not found or not readable), raw_text, ocr_conf (0-1), box (x1, y1, x2, y2) or None."""
    result = plate_model.predict(image[:, :, ::-1], conf=PLATE_CONF, iou=PLATE_NMS_IOU, imgsz=PLATE_IMGSZ, verbose=False)[0]  # YOLO expects BGR arrays
    if len(result.boxes) == 0:
        return {"plate": "", "raw_text": "", "ocr_conf": 0.0, "box": None}

    box = result.boxes[int(result.boxes.conf.argmax())]  # the most confident plate
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    crop = image[max(0, y1 - 3):y2 + 3, max(0, x1 - 3):x2 + 3]
    raw_text, ocr_conf = read_plate(crop)
    plate = clean_plate(raw_text)
    return {"plate": format_plate(plate) if plate else "", "raw_text": raw_text, "ocr_conf": ocr_conf, "box": (x1, y1, x2, y2)}


# Quick check: read the plate of every car found in the Step 14 image
image_rgb = load_rgb(img_path)
for obj in detect_parking(image_rgb):
    if obj["class"] == "Empty":
        continue
    x1, y1, x2, y2 = (int(v) for v in obj["box"])
    found = read_plate_number(image_rgb[max(0, y1):y2, max(0, x1):x2])
    print(f"{obj['class']:<9} plate: {found['plate'] or '(not found)'}")

## Step 19: Notify the owner of a double-parked car

**Feature 1.** When the model classifies a car as `Illegal` (double-parked in the aisle, blocking
a legally parked car), the car park should notify that car's owner.

This step turns each `Illegal` detection into a structured **notification event** that a backend
can send out (SMS / email / push). What is machine learning here and what is not:

- **Machine learning (done by the model):** finding the double-parked car (the `Illegal` box).
- **Machine learning (plate model, Step 18):** reading the offender's plate. The `Illegal` box is cut
  out of the image and passed to `read_plate_number(image)`, so `plate` below is filled in automatically
  (`None` when the plate is too small or blurred to read).
- **Not machine learning (backend):** the actual sending of the message and the plate -> owner lookup
  (name, phone, email) live in a database and a messaging service, not in the model. This step only
  produces the event; the dashboard/backend team consumes it.

The event is printed as JSON so it can be dropped straight onto a message queue or REST endpoint.

In [ ]:
import json
from datetime import datetime

# --- Plate lookup ------------------------------------------------------------
# The offender's plate is read from the Illegal box with read_plate_number() (Step 18).
# Set USE_PLATE_MODEL = False to send the events without a plate.
USE_PLATE_MODEL = True

def owner_plate_for(car_box, image=None):
    """Return the licence plate of the car in car_box, or None if it cannot be read."""
    if USE_PLATE_MODEL and image is not None:
        x1, y1, x2, y2 = (int(v) for v in car_box)
        crop = image[max(0, y1):y2, max(0, x1):x2]
        return read_plate_number(crop)["plate"] or None  # "" means no plate found or not readable
    return None

CARPARK_LOCATION = "Level 1, Car Park A"  # describes this camera/zone for the dashboard

def build_notification_events(detections, image=None, location=CARPARK_LOCATION):
    """detections: output of detect_parking(). Returns one notify event per double-parked car."""
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    events = []
    for i, obj in enumerate(d for d in detections if d["class"] == "Illegal"):
        plate = owner_plate_for(obj["box"], image)
        events.append({
            "event": "double_parking_detected",
            "vehicle_id": f"V{i + 1:03d}",
            "plate": plate,                 # None if the plate could not be read
            "owner_lookup": "pending" if plate is None else "resolve_in_backend",
            "location": location,
            "box": [round(v) for v in obj["box"]],
            "confidence": round(obj["conf"], 3),
            "detected_at": now,
            "message": "Your vehicle is double-parked and blocking the aisle. Please move it.",
            "penalty": "RM 50 (simulated summons)",
            "notify_channel": ["sms", "email", "push"],  # backend decides what to send
        })
    return events

# Build events from the Step 14 image detections
image_rgb = load_rgb(img_path)
parking_detections = detect_parking(image_rgb)
notify_events = build_notification_events(parking_detections, image_rgb)

print(f"Double-parked cars found: {len(notify_events)}")
print(json.dumps(notify_events, indent=2))
if not notify_events:
    print("\n(No Illegal car in this image - try re-running Step 14 for a photo that has one.)")

## Step 20: Dashboard feed - virtual parking boxes

**Feature 2.** The dashboard shows a virtual box for every parking area so a user can see, at a
glance, what is free and what is not, and decide where to park:

- **Green** box  -> `Empty`: the slot is free.
- **Red** box    -> `Occupied`: a car is parked there.
- **Illegal** marker (drawn in orange here) -> a car double-parked in front / in the aisle.

This is **not a machine-learning model** - it is a visualisation. The model's job is only to produce
the detections; this step converts them into a small, stable **JSON contract** the dashboard front-end
can render directly (web, mobile, etc.). The preview image below shows what that feed looks like, but the
real interactive dashboard (clickable boxes, "should I park here?" suggestion) is a separate front-end
app that consumes this JSON.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Status -> colour contract shared with the dashboard front-end
STATUS_COLOR = {"Empty": "#21c25e", "Occupied": "#e03131", "Illegal": "#f08c00"}
STATUS_LABEL = {"Empty": "Free", "Occupied": "Occupied", "Illegal": "Double-parked"}

def build_dashboard_feed(detections, location=CARPARK_LOCATION):
    """Turn detect_parking() output into a dashboard-ready feed."""
    boxes = []
    for i, obj in enumerate(detections):
        cls = obj["class"]
        x1, y1, x2, y2 = (round(v) for v in obj["box"])
        boxes.append({
            "id": f"B{i + 1:03d}",
            "status": cls,                       # Empty | Occupied | Illegal
            "label": STATUS_LABEL[cls],
            "color": STATUS_COLOR[cls],
            "parkable": cls == "Empty",          # drives the 'can I park here?' hint
            "confidence": round(obj["conf"], 3),
            "box": [x1, y1, x2, y2],             # pixel coords for overlay on the camera image
        })
    summary = {
        "empty": sum(b["status"] == "Empty" for b in boxes),
        "occupied": sum(b["status"] == "Occupied" for b in boxes),
        "illegal": sum(b["status"] == "Illegal" for b in boxes),
    }
    return {
        "location": location,
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "summary": summary,
        "boxes": boxes,
    }

dashboard_feed = build_dashboard_feed(parking_detections)
print(json.dumps(dashboard_feed, indent=2))

# Preview: draw the virtual boxes on the camera image the way the dashboard would
plt.figure(figsize=(8, 8))
ax = plt.gca()
ax.imshow(image_rgb)
for b in dashboard_feed["boxes"]:
    x1, y1, x2, y2 = b["box"]
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=b["color"], linewidth=3))
    ax.text(x1, y1 - 4, b["label"], color=b["color"], fontsize=9, weight="bold")
s = dashboard_feed["summary"]
plt.title(f"Dashboard preview  |  Free: {s['empty']}  Occupied: {s['occupied']}  Double-parked: {s['illegal']}")
plt.axis("off")
plt.show()

# Save the feed so the dashboard team has a concrete sample to build against
with open("dashboard_feed_sample.json", "w") as f:
    json.dump(dashboard_feed, f, indent=2)
print("\nSaved sample feed to dashboard_feed_sample.json")

## Step 21: Convert both models for the Edge TPU

The Coral Edge TPU cannot run `.pt` files. Each model is converted once into an 8-bit `_edgetpu.tflite` file, saved in `datasets/models`:

- Run this step on Colab: the Edge TPU compiler only works on Linux.
- Copy the two `.tflite` files and `edge_tpu_detection.py` to the Dev Board Mini (see the README), then run `python3 edge_tpu_detection.py photo.jpg`.
- Nothing is trained on the device. Convert again only after the models are retrained.

**Why a separate Python.** Colab's own Python is too new: its TensorFlow version makes Edge TPU files where almost the whole model runs on the board's CPU instead of the Edge TPU (1 of 256 operations on the Edge TPU). This step therefore converts in a separate Python 3.11 with TensorFlow 2.19, the setup that was tested (almost every operation on the Edge TPU). It checks the result and stops if most of a model would run on the CPU.

The 8-bit model is slightly less accurate than the `.pt` model, so the parking model's test mAP50 is checked at the end.

In [ ]:
# Convert both models for the Coral Edge TPU (8-bit), in a separate Python 3.11 (see above). Run on Colab.
import os
import shutil
import subprocess

TPU_PYTHON = "/content/tpu_env/bin/python"
if not os.path.exists(TPU_PYTHON):
    !pip install -q uv
    !uv venv -q -p 3.11 /content/tpu_env
    !uv pip install -q -p {TPU_PYTHON} "ultralytics[export]==8.4.182" "tensorflow==2.19.0" "tf_keras==2.19.0" "onnx2tf==1.28.8" "sng4onnx==2.0.1" "onnx_graphsurgeon==0.6.1" pip torch torchvision --extra-index-url https://download.pytorch.org/whl/cpu --index-strategy unsafe-best-match

# The 8-bit conversion measures typical values on sample images: car park photos for the parking model,
# car photos for the plate model (only images are needed, no labels)
PLATE_CALIBRATION = "plate_calibration.yaml"
with open(PLATE_CALIBRATION, "w") as f:
    f.write(f"path: {os.path.abspath('datasets/car-plate-data')}\ntrain: valid\nval: valid\nnames:\n  0: LicensePlate\n")

with open("export_edgetpu.py", "w") as f:
    f.write("import sys\nfrom ultralytics import YOLO\n"
            "print(YOLO(sys.argv[1]).export(format='edgetpu', imgsz=512, data=sys.argv[2]))\n")

env = dict(os.environ, PATH=os.path.dirname(TPU_PYTHON) + ":" + os.environ["PATH"])
EDGE_TPU_FILES = {}
for pt_file, calibration in [(PARKING_MODEL_FILE, DATA_YAML), (PLATE_MODEL_FILE, PLATE_CALIBRATION)]:
    run = subprocess.run([TPU_PYTHON, "export_edgetpu.py", pt_file, calibration], env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    exported = run.stdout.strip().splitlines()[-1]
    assert exported.endswith("_edgetpu.tflite"), run.stdout[-3000:]

    # The Edge TPU compiler log lists every operation and where it runs
    log_lines = [line.split() for line in open(exported.replace(".tflite", ".log"))]
    counts = [(int(p[1]), "Mapped to Edge TPU" in " ".join(p)) for p in log_lines if len(p) > 2 and p[1].isdigit()]
    on_tpu = sum(n for n, mapped in counts if mapped)
    on_cpu = sum(n for n, mapped in counts if not mapped)
    print(f"{os.path.basename(pt_file)}: {on_tpu} operations on the Edge TPU, {on_cpu} on the CPU")
    assert on_tpu > 10 * on_cpu, "Most of the model would run on the CPU: check the Python and TensorFlow versions above"

    EDGE_TPU_FILES[pt_file] = pt_file.replace(".pt", "_edgetpu.tflite")  # the name edge_tpu_detection.py expects
    shutil.copy(exported, EDGE_TPU_FILES[pt_file])
    print("Saved", EDGE_TPU_FILES[pt_file])

    if pt_file == PARKING_MODEL_FILE:
        # The _edgetpu file only runs on the Edge TPU, so check the plain 8-bit file made next to it (same numbers)
        int8_file = exported.replace("_edgetpu.tflite", ".tflite")
        edge_metrics = YOLO(int8_file, task="detect").val(data=DATA_YAML, split="test", imgsz=512, verbose=False)
        print(f"Parking test mAP50: .pt {metrics.box.map50:.3f} | 8-bit {edge_metrics.box.map50:.3f}")

try:
    from google.colab import files
    for edge_file in EDGE_TPU_FILES.values():
        files.download(edge_file)
except ImportError:
    pass  # running locally, the files are already in datasets/models